# 02 — Data cleaning and validation
Scope: the two initial placement rounds only. Validate every raw JSON against the extracted CSVs. The desert is a valid null, not a missing observation. Run from the repository root.

In [ ]:
from pathlib import Path
import json, ast
from collections import Counter
import pandas as pd

DATA = Path("data_5000")
OUT = Path("processed_data_5000")
p = pd.read_csv(OUT / "initial_placements.csv")
m = pd.read_csv(OUT / "map_tiles.csv")
files = sorted(DATA.glob("*.json"))
assert files, "No raw JSON files found"
assert not p.duplicated(["Game_ID", "Player"]).any()
assert not m.duplicated(["Game_ID", "Tile_ID"]).any()
assert not p.isna().any().any()
assert p.groupby("Game_ID").size().eq(4).all()
assert m.groupby("Game_ID").size().eq(19).all()
assert set(p.Game_ID) == set(m.Game_ID)
assert m.Resource.isna().equals(m.Number.isna())
assert m.groupby("Game_ID").Resource.apply(lambda s: s.isna().sum()).eq(1).all()
assert (m.Coordinate_X + m.Coordinate_Y + m.Coordinate_Z).eq(0).all()
print(len(files), len(p), len(m))

In [ ]:
expected_p, expected_m = [], []
seen = set()
for file in files:
    g = json.loads(file.read_text(encoding="utf-8"))
    gid, colors = g["game"]["id"], g["colors"]
    assert gid not in seen and len(set(colors)) == 4, gid
    assert g["map"]["template"] == "BASE", gid
    seen.add(gid)
    a = [r[0] for r in g["action_records"][:16]]
    order = colors + colors[::-1]
    assert len(a) == 16, gid
    for j, color in enumerate(order):
        s, r = a[2*j:2*j+2]
        assert s[:2] == [color, "BUILD_SETTLEMENT"], gid
        assert r[:2] == [color, "BUILD_ROAD"], gid
        assert s[2] in r[2], gid
    for i, color in enumerate(colors, 1):
        actions = [r for r in a if r[0] == color]
        expected_p.append([gid, color, i, actions[0][2],
                           str(actions[1][2]), actions[2][2], str(actions[3][2])])
    land = [t for t in g["map"]["tiles"] if t["type"] == "LAND"]
    assert len(land) == 19, gid
    assert Counter(t["resource"] for t in land) == Counter(
        {"WOOD":4,"BRICK":3,"SHEEP":4,"WHEAT":4,"ORE":3,None:1}), gid
    assert sorted(t["number"] for t in land if t["number"] is not None) == [
        2,3,3,4,4,5,5,6,6,8,8,9,9,10,10,11,11,12], gid
    for t in land:
        expected_m.append([gid,t["id"],*t["coordinate"],t["resource"],t["number"]])
assert seen == set(p.Game_ID)
def canonical(frame, keys):
    return frame.sort_values(keys).reset_index(drop=True)
ep = pd.DataFrame(expected_p, columns=p.columns)
em = pd.DataFrame(expected_m, columns=m.columns)
for frame in (ep, p):
    for col in ["Road_1", "Road_2"]:
        frame[col] = frame[col].map(lambda s: str(sorted(ast.literal_eval(s))))
pd.testing.assert_frame_equal(canonical(ep,["Game_ID","Player"]),
    canonical(p,["Game_ID","Player"]), check_dtype=False)
pd.testing.assert_frame_equal(canonical(em,["Game_ID","Tile_ID"]),
    canonical(m,["Game_ID","Tile_ID"]), check_dtype=False)
print(f"Validated {len(seen)} games against original setup records.")

In [ ]:
from importlib import import_module
fe = import_module("03_feature_engineering")
nodes, edges, neighbors = fe.build_topology(m)
for gid, group in p.groupby("Game_ID"):
    occupied = set(group.Settlement_1) | set(group.Settlement_2)
    assert len(occupied) == 8 and occupied <= set(nodes), gid
    assert not any(a in occupied and b in occupied for a,b in edges), gid
    roads = {tuple(sorted(ast.literal_eval(r)))
             for col in ["Road_1","Road_2"] for r in group[col]}
    assert len(roads) == 8 and roads <= edges, gid
print("Board topology and distance-rule checks passed.")
display(pd.DataFrame({"placements":p.isna().sum(),"map_tiles":m.isna().sum()}))
# No imputation or deletion is required. Preserve the original CSV files.